In [2]:
from pathlib import Path

# Always anchor paths to project root
PROJECT_ROOT = Path.cwd()

# If running inside notebooks/02_preprocess, go up two levels
if PROJECT_ROOT.name == "02_preprocess":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
elif PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: c:\Users\User\Desktop\Vehicle_Damage_Detection


In [1]:
from pathlib import Path
from PIL import Image

# ===================== SETTINGS =====================
TARGET_SIZE = (224, 224)

def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    while cur != cur.parent:
        if (cur / ".dvc").exists() or (cur / ".git").exists():
            return cur
        cur = cur.parent
    raise FileNotFoundError("Could not find project root (no .dvc or .git found).")

PROJECT_ROOT = find_project_root(Path.cwd())

RAW_ROOT = PROJECT_ROOT / "data" / "raw" / "CarDD_release" / "CarDD_release" / "CarDD_SOD"
OUT_ROOT = PROJECT_ROOT / "data" / "processed" / "cardd_sod"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_ROOT:", RAW_ROOT, "| exists:", RAW_ROOT.exists())
print("OUT_ROOT:", OUT_ROOT)

# ===================== SPLIT MAPPING (YOUR CASE) =====================
SPLIT_MAP = {
    "train": RAW_ROOT / "CarDD-TR",
    "val":   RAW_ROOT / "CarDD-VAL",
    "test":  RAW_ROOT / "CarDD-TE",
}

for k, v in SPLIT_MAP.items():
    print(f"{k.upper()} split path:", v, "| exists:", v.exists())

if not all(p.exists() for p in SPLIT_MAP.values()):
    raise FileNotFoundError("One of CarDD-TR / CarDD-VAL / CarDD-TE is missing. Check folder names again.")

# ===================== HELPERS =====================
def find_images_and_masks(split_dir: Path):
    """
    Inside CarDD-TR / CarDD-VAL / CarDD-TE
    find a folder that looks like images and one that looks like masks.
    """
    img_exts = {".jpg", ".jpeg", ".png"}
    mask_exts = {".png"}

    subdirs = [p for p in split_dir.iterdir() if p.is_dir()]
    if not subdirs:
        raise FileNotFoundError(f"No subfolders in {split_dir}")

    def count_ext(d, exts):
        return sum(1 for f in d.iterdir() if f.is_file() and f.suffix.lower() in exts)

    # image folder = most jpg/jpeg/png
    img_dir = max(subdirs, key=lambda d: count_ext(d, img_exts))

    # mask folder = most png (try avoid same as img_dir)
    ranked = sorted(subdirs, key=lambda d: count_ext(d, mask_exts), reverse=True)
    mask_dir = ranked[0]
    if mask_dir == img_dir and len(ranked) > 1:
        mask_dir = ranked[1]

    return img_dir, mask_dir

def resize_image(src: Path, dst: Path):
    with Image.open(src) as im:
        im = im.convert("RGB")
        im = im.resize(TARGET_SIZE, resample=Image.BILINEAR)
        dst.parent.mkdir(parents=True, exist_ok=True)
        im.save(dst, format="JPEG", quality=95)

def resize_mask(src: Path, dst: Path):
    with Image.open(src) as m:
        m = m.resize(TARGET_SIZE, resample=Image.NEAREST)
        dst.parent.mkdir(parents=True, exist_ok=True)
        m.save(dst, format="PNG")

# ===================== RUN PREPROCESS =====================
OUT_ROOT.mkdir(parents=True, exist_ok=True)

total_img = 0
total_mask = 0

for split in ["train", "val", "test"]:
    split_dir = SPLIT_MAP[split]

    img_dir, mask_dir = find_images_and_masks(split_dir)

    print(f"\n--- {split.upper()} ---")
    print("Split dir :", split_dir)
    print("Images dir:", img_dir)
    print("Masks dir :", mask_dir)

    out_img_dir = OUT_ROOT / split / "images"
    out_mask_dir = OUT_ROOT / split / "masks"
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_mask_dir.mkdir(parents=True, exist_ok=True)

    img_files = sorted([f for f in img_dir.iterdir()
                        if f.is_file() and f.suffix.lower() in [".jpg", ".jpeg", ".png"]])

    processed_img = 0
    processed_mask = 0
    missing_mask = 0

    for img_path in img_files:
        stem = img_path.stem

        # find corresponding mask by same stem
        mask_path = None
        for ext in [".png", ".jpg", ".jpeg"]:
            cand = mask_dir / f"{stem}{ext}"
            if cand.exists():
                mask_path = cand
                break

        out_img_path = out_img_dir / f"{stem}.jpg"
        out_mask_path = out_mask_dir / f"{stem}.png"

        resize_image(img_path, out_img_path)
        processed_img += 1

        if mask_path is None:
            missing_mask += 1
        else:
            resize_mask(mask_path, out_mask_path)
            processed_mask += 1

    print(" Images processed:", processed_img)
    print(" Masks processed :", processed_mask)
    print(" Missing masks   :", missing_mask)

    total_img += processed_img
    total_mask += processed_mask

print("\n.. PREPROCESS DONE")
print("Saved to:", OUT_ROOT)
print("Total images:", total_img)
print("Total masks :", total_mask)


PROJECT_ROOT: C:\Users\User\Desktop\Vehicle_Damage_Detection
RAW_ROOT: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\raw\CarDD_release\CarDD_release\CarDD_SOD | exists: True
OUT_ROOT: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\cardd_sod
TRAIN split path: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\raw\CarDD_release\CarDD_release\CarDD_SOD\CarDD-TR | exists: True
VAL split path: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\raw\CarDD_release\CarDD_release\CarDD_SOD\CarDD-VAL | exists: True
TEST split path: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\raw\CarDD_release\CarDD_release\CarDD_SOD\CarDD-TE | exists: True

--- TRAIN ---
Split dir : C:\Users\User\Desktop\Vehicle_Damage_Detection\data\raw\CarDD_release\CarDD_release\CarDD_SOD\CarDD-TR
Images dir: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\raw\CarDD_release\CarDD_release\CarDD_SOD\CarDD-TR\CarDD-TR-Edge
Masks dir : C:\Users\User\Desktop\Vehicle_Damage_Detection\data\raw\CarDD

In [2]:
from pathlib import Path

p = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\cardd_sod")

def count_files(folder, exts):
    return sum(1 for f in folder.rglob("*") if f.is_file() and f.suffix.lower() in exts)

for split in ["train", "val", "test"]:
    img_dir = p / split / "images"
    mask_dir = p / split / "masks"
    print(f"\n{split.upper()}")
    print(" images:", count_files(img_dir, {".jpg", ".jpeg", ".png"}))
    print(" masks :", count_files(mask_dir, {".png"}))



TRAIN
 images: 2816
 masks : 2816

VAL
 images: 810
 masks : 810

TEST
 images: 374
 masks : 374


In [2]:
from pathlib import Path

base1 = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\cardd_sod")
base2 = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\cardd_sod_yolo")

print("cardd_sod exists:", base1.exists())
if base1.exists():
    print("\ncardd_sod contents:")
    for p in base1.iterdir():
        print("-", p.name)

print("\ncardd_sod_yolo exists:", base2.exists())
if base2.exists():
    print("\ncardd_sod_yolo contents:")
    for p in base2.iterdir():
        print("-", p.name)

cardd_sod exists: True

cardd_sod contents:
- test
- train
- val

cardd_sod_yolo exists: False


In [3]:
from pathlib import Path

BASE = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\cardd_sod")

for split in ["train", "val", "test"]:
    img_dir = BASE / split / "images"
    mask_dir = BASE / split / "masks"

    imgs = list(img_dir.glob("*")) if img_dir.exists() else []
    masks = list(mask_dir.glob("*")) if mask_dir.exists() else []

    print(f"\n=== {split.upper()} ===")
    print("images:", len(imgs))
    print("masks :", len(masks))
    print("sample image names:", [f.name for f in imgs[:3]])
    print("sample mask names :", [f.name for f in masks[:3]])


=== TRAIN ===
images: 2816
masks : 2816
sample image names: ['000001.jpg', '000002.jpg', '000003.jpg']
sample mask names : ['000001.png', '000002.png', '000003.png']

=== VAL ===
images: 810
masks : 810
sample image names: ['000013.jpg', '000016.jpg', '000017.jpg']
sample mask names : ['000013.png', '000016.png', '000017.png']

=== TEST ===
images: 374
masks : 374
sample image names: ['000012.jpg', '000015.jpg', '000023.jpg']
sample mask names : ['000012.png', '000015.png', '000023.png']


In [4]:
from pathlib import Path

BASE = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\cardd_sod")

for split in ["train", "val", "test"]:
    img_dir = BASE / split / "images"
    mask_dir = BASE / split / "masks"

    img_stems = {f.stem for f in img_dir.glob("*")}
    mask_stems = {f.stem for f in mask_dir.glob("*")}

    common = img_stems & mask_stems
    only_imgs = img_stems - mask_stems
    only_masks = mask_stems - img_stems

    print(f"\n=== {split.upper()} ===")
    print("matching pairs:", len(common))
    print("images without masks:", len(only_imgs))
    print("masks without images:", len(only_masks))

    if only_imgs:
        print("sample images without masks:", list(sorted(only_imgs))[:5])
    if only_masks:
        print("sample masks without images:", list(sorted(only_masks))[:5])


=== TRAIN ===
matching pairs: 2816
images without masks: 0
masks without images: 0

=== VAL ===
matching pairs: 810
images without masks: 0
masks without images: 0

=== TEST ===
matching pairs: 374
images without masks: 0
masks without images: 0


In [5]:
from pathlib import Path
from PIL import Image
import numpy as np

BASE = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\cardd_sod")
mask_dir = BASE / "train" / "masks"

all_vals = set()

for mf in list(mask_dir.glob("*"))[:20]:
    arr = np.array(Image.open(mf))
    all_vals.update(np.unique(arr).tolist())

print("Unique mask values:", sorted(all_vals))

Unique mask values: [0, 255]


In [6]:
from pathlib import Path
from PIL import Image
import numpy as np
import shutil

BASE = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\cardd_sod")
OUT = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\cardd_sod_yolo")

# create output folders
for split in ["train", "val", "test"]:
    (OUT / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT / "labels" / split).mkdir(parents=True, exist_ok=True)

def mask_to_yolo_polygon(mask_path: Path):
    """
    Convert a binary mask into one YOLO segmentation polygon line.
    Class id is fixed as 0 (single class: damage).
    """
    arr = np.array(Image.open(mask_path).convert("L"))
    ys, xs = np.where(arr > 0)

    # no object found
    if len(xs) == 0 or len(ys) == 0:
        return None

    h, w = arr.shape

    # simple bounding polygon from mask extent
    x_min, x_max = xs.min(), xs.max()
    y_min, y_max = ys.min(), ys.max()

    # normalize to 0-1
    x1 = x_min / w
    y1 = y_min / h
    x2 = x_max / w
    y2 = y_min / h
    x3 = x_max / w
    y3 = y_max / h
    x4 = x_min / w
    y4 = y_max / h

    # YOLO segmentation format: class x1 y1 x2 y2 x3 y3 ...
    return f"0 {x1:.6f} {y1:.6f} {x2:.6f} {y2:.6f} {x3:.6f} {y3:.6f} {x4:.6f} {y4:.6f}"

for split in ["train", "val", "test"]:
    img_dir = BASE / split / "images"
    mask_dir = BASE / split / "masks"

    out_img_dir = OUT / "images" / split
    out_lab_dir = OUT / "labels" / split

    copied = 0
    labels_written = 0
    skipped = 0

    for img_path in img_dir.glob("*"):
        mask_path = mask_dir / f"{img_path.stem}.png"
        if not mask_path.exists():
            skipped += 1
            continue

        # copy image
        shutil.copy2(img_path, out_img_dir / img_path.name)
        copied += 1

        # write label
        label_line = mask_to_yolo_polygon(mask_path)
        label_path = out_lab_dir / f"{img_path.stem}.txt"

        if label_line is None:
            label_path.write_text("")
        else:
            label_path.write_text(label_line)
            labels_written += 1

    print(f"{split}: copied={copied}, labels_written={labels_written}, skipped={skipped}")

# create data.yaml
yaml_text = f"""
path: {OUT.as_posix()}
train: images/train
val: images/val
test: images/test

names:
  0: damage
""".strip()

(OUT / "data.yaml").write_text(yaml_text, encoding="utf-8")

print("\nCreated:", OUT)
print("data.yaml exists:", (OUT / "data.yaml").exists())

train: copied=2816, labels_written=2816, skipped=0
val: copied=810, labels_written=810, skipped=0
test: copied=374, labels_written=374, skipped=0

Created: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\cardd_sod_yolo
data.yaml exists: True


In [7]:
from pathlib import Path

OUT = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\cardd_sod_yolo")

for split in ["train", "val", "test"]:
    imgs = list((OUT / "images" / split).glob("*"))
    labs = list((OUT / "labels" / split).glob("*.txt"))

    print(f"\n=== {split.upper()} ===")
    print("images:", len(imgs))
    print("labels:", len(labs))

print("\nYAML exists:", (OUT / "data.yaml").exists())


=== TRAIN ===
images: 2816
labels: 2816

=== VAL ===
images: 810
labels: 810

=== TEST ===
images: 374
labels: 374

YAML exists: True
